In [1]:
# ── Cell 1: Mount Drive & clone repo ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from getpass import getpass
import subprocess, os

token    = getpass("GitHub token: ")
repo_dir = '/content/notebooks_meta'

if not os.path.exists(repo_dir):
    subprocess.run(
        ['git', 'clone',
         f'https://{token}@github.com/jonyxdxp/notebooks_meta.git',
         repo_dir], check=True)

%cd {repo_dir}
!git pull origin main

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GitHub token: ··········
/content/notebooks_meta
From https://github.com/jonyxdxp/notebooks_meta
 * branch            main       -> FETCH_HEAD
Already up to date.


In [2]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 511.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# !wget http://yanran.li/files/ijcnlp_dailydialog.zip -P /content/drive/MyDrive/data/

In [4]:
# ── Cell 3: Extract raw DailyDialog data (skip if already done) ──────────────
import zipfile

zip_path        = '/content/drive/MyDrive/data/dialog_data/ijcnlp_dailydialog.zip'
extraction_path = f'{repo_dir}/v3/first_phase/data/raw'
os.makedirs(extraction_path, exist_ok=True)

if not os.path.exists(os.path.join(extraction_path, 'ijcnlp_dailydialog')):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extraction_path)
    print(f"Extracted to: {extraction_path}")
else:
    print("Data already extracted, skipping.")

Extracted to: /content/notebooks_meta/v3/first_phase/data/raw


In [5]:
# ── Cell 4: Imports ───────────────────────────────────────────────────────────
import math, os, time
import sys

import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm

# All our modules live here
sys.path.insert(0, f'{repo_dir}/v3/first_phase')



In [6]:
# ── Cell 6: Device ────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if device.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [7]:
# --------------------------------------------------------------------------
# Tokenizer
# ---------------------------------------------------------------------------

def get_tokenizer(config):
  tokenizer = transformers.AutoTokenizer.from_pretrained(
    config.data.tokenizer_name_or_path)

  if (isinstance(tokenizer, transformers.GPT2TokenizerFast)
      or isinstance(tokenizer, transformers.GPT2Tokenizer)):
    tokenizer._tokenizer.post_processor = tokenizers.processors.BertProcessing(
      (tokenizer.bos_token, tokenizer.bos_token_id),
      (tokenizer.eos_token, tokenizer.eos_token_id))

  if tokenizer.bos_token is None:
    if tokenizer.cls_token is None:
      raise AttributeError(
        'Tokenizer must have a bos_token or '
        f'cls_token: {tokenizer}')
    tokenizer.bos_token = tokenizer.cls_token
  if tokenizer.eos_token is None:
    if tokenizer.sep_token is None:
      raise AttributeError(
        'Tokenizer must have a eos_token or '
        f'sep_token: {tokenizer}')
    tokenizer.eos_token = tokenizer.sep_token
  if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

  return tokenizer




In [8]:
# dataset

import datasets

# ---------------------------------------------------------------------------
# DailyDialog dataset loader
# ---------------------------------------------------------------------------

def get_dailydialog_dataset(
    cache_dir: str,
    tokenizer,
    block_size: int = 128,
    num_proc: int = 1,
    raw_data_dir: str = '/content/notebooks_meta/v3/first_phase/data/raw/ijcnlp_dailydialog',
) -> datasets.DatasetDict:
  """Load DailyDialog from local files and flatten every turn into its own sample.

  Handles the ijcnlp_dailydialog zip layout where each split ships as its own
  zip (train.zip / validation.zip / test.zip) containing a dialogues_*.txt file.
  Falls back to the combined dialogues_text.txt if split zips are unavailable.
  """
  import zipfile as _zipfile

  _cache_path = os.path.join(cache_dir, f'dailydialog_jepa_bs{block_size}')
  if os.path.exists(_cache_path):
    print(f'Loading from cache: {_cache_path}')
    return datasets.load_from_disk(_cache_path).with_format('torch')

  # ── Step 1: ensure the individual txt files are extracted ───────────────
  # Each zip contains one file named e.g. dialogues_train.txt
  split_zips = {
    'train':      os.path.join(raw_data_dir, 'train.zip'),
    'validation': os.path.join(raw_data_dir, 'validation.zip'),
    'test':       os.path.join(raw_data_dir, 'test.zip'),
  }
  split_txts = {
    'train':      os.path.join(raw_data_dir, 'train', 'dialogues_train.txt'),
    'validation': os.path.join(raw_data_dir, 'validation', 'dialogues_validation.txt'),
    'test':       os.path.join(raw_data_dir, 'test', 'dialogues_test.txt'),
  }

  for split, zip_path in split_zips.items():
    txt_path = split_txts[split]
    if os.path.exists(txt_path):
      continue                          # already extracted
    if not os.path.exists(zip_path):
      raise FileNotFoundError(
        f'Neither {txt_path} nor {zip_path} found.\n'
        f'Contents of {raw_data_dir}: {os.listdir(raw_data_dir)}')
    print(f'  Extracting {os.path.basename(zip_path)} ...')
    with _zipfile.ZipFile(zip_path, 'r') as zf:
      zf.extractall(raw_data_dir)
    if not os.path.exists(txt_path):
      # zip may contain the file under a different name — show what came out
      raise FileNotFoundError(
        f'Extraction succeeded but {txt_path} still not found.\n'
        f'Contents now: {os.listdir(raw_data_dir)}')

  # ── Step 2: read utterances ──────────────────────────────────
  def _load_utterances(filepath: str):
    utterances = []
    with open(filepath, 'r', encoding='utf-8') as f:
      for line in f:
        line = line.strip()
        if not line:
          continue
        turns = [t.strip() for t in line.split('__eou__') if t.strip()]
        utterances.extend(turns)
    return utterances

  def _tokenize(examples):
    return tokenizer(
      examples['text'],
      max_length=block_size,
      padding='max_length',
      truncation=True,
      add_special_tokens=True,
      return_attention_mask=True,
      return_token_type_ids=False,
    )

  print('Building DailyDialog JEPA dataset ...')
  tokenized_splits = {}

  for split, txt_path in split_txts.items():
    utterances = _load_utterances(txt_path)
    print(f'  {split:12s}: {len(utterances):>6,} utterances')
    raw_ds = datasets.Dataset.from_dict({'text': utterances})
    tokenized_splits[split] = raw_ds.map(
      _tokenize,
      batched=True,
      num_proc=num_proc,
      remove_columns=['text'],
      desc=f'Tokenizing {split}',
    )

  dataset_dict = datasets.DatasetDict(tokenized_splits)
  os.makedirs(cache_dir, exist_ok=True)
  dataset_dict.save_to_disk(_cache_path)
  print(f'Saved to cache: {_cache_path}')
  return dataset_dict.with_format('torch')

In [9]:
# data collator

import typing

# ---------------------------------------------------------------------------
# JEPA mask collator
# ---------------------------------------------------------------------------

class JEPAMaskCollator:
  """Produces context / target view pairs for JEPA phase-1 training.

  Both views come from the same tokenized sequence:
    - target_input_ids:   clean, unmasked sequence  → target encoder (stop-grad)
    - context_input_ids:  target spans replaced with [MASK] → context encoder
    - target_mask:        bool (B, L), True at masked positions → predictor loss

  Span sampling: ``num_target_spans`` non-overlapping spans of
  ``target_span_length`` tokens, sampled uniformly from the maskable region
  (excluding BOS, EOS, and padding).
  """

  def __init__(
      self,
      mask_token_id: int,
      pad_token_id: int,
      num_target_spans: int = 4,
      target_span_length: int = 8,
  ):
    self.mask_token_id = mask_token_id
    self.pad_token_id = pad_token_id
    self.num_target_spans = num_target_spans
    self.target_span_length = target_span_length

  def __call__(
      self,
      batch: typing.List[typing.Dict[str, torch.Tensor]],
  ) -> typing.Dict[str, torch.Tensor]:
    input_ids      = torch.stack([b['input_ids']      for b in batch])
    attention_mask = torch.stack([b['attention_mask'] for b in batch])
    B, L = input_ids.shape

    target_input_ids  = input_ids.clone()
    context_input_ids = input_ids.clone()
    target_mask = torch.zeros(B, L, dtype=torch.bool)

    for i in range(B):
      valid_len = int(attention_mask[i].sum().item())
      # maskable region: skip BOS (idx 0) and EOS (last real token)
      spans = self._sample_spans(maskable_start=1, maskable_end=valid_len - 1)
      for s, e in spans:
        target_mask[i, s:e] = True
      context_input_ids[i, target_mask[i]] = self.mask_token_id

    return {
      'context_input_ids':      context_input_ids,   # (B, L)
      'context_attention_mask': attention_mask,       # (B, L)
      'target_input_ids':       target_input_ids,     # (B, L)
      'target_attention_mask':  attention_mask,       # (B, L)
      'target_mask':            target_mask,          # (B, L) bool
    }

  def _sample_spans(
      self,
      maskable_start: int,
      maskable_end: int,
  ) -> typing.List[typing.Tuple[int, int]]:
    region_len = maskable_end - maskable_start
    if region_len <= 0:
      return []

    span_len  = min(self.target_span_length, region_len)
    available = list(range(maskable_start, maskable_end - span_len + 1))

    spans: typing.List[typing.Tuple[int, int]] = []
    for _ in range(self.num_target_spans):
      if not available:
        break
      idx = torch.randint(len(available), (1,)).item()
      s, e = available[idx], available[idx] + span_len
      spans.append((s, e))
      available = [x for x in available if (x + span_len <= s) or (x >= e)]

    return spans

In [10]:
# dataloader


# ---------------------------------------------------------------------------
# JEPA dataloaders
# ---------------------------------------------------------------------------

def get_jepa_dataloaders(
    config,
    tokenizer,
    skip_train: bool = False,
    skip_valid: bool = False,
    valid_seed: typing.Optional[int] = None,
):
  """Build train / validation DataLoaders for JEPA phase-1 training.

  Expects config.jepa.num_target_spans and config.jepa.target_span_length
  (defaults: 4 and 8).  Batch items have keys produced by JEPAMaskCollator.
  """
  num_gpus  = max(torch.cuda.device_count(), 1)
  block_size = config.model.length

  if config.loader.eval_global_batch_size % num_gpus != 0:
    raise ValueError(
      f'Eval batch size {config.loader.eval_global_batch_size} '
      f'not divisible by {num_gpus} GPUs.')

  dataset_dict = get_dailydialog_dataset(
    cache_dir=config.data.cache_dir,
    tokenizer=tokenizer,
    block_size=block_size,
  )

  if tokenizer.mask_token_id is None:
    raise ValueError(
      f'Tokenizer must have a mask_token for JEPA masking: {tokenizer}')

  jepa_cfg = getattr(config, 'jepa', None)
  collator = JEPAMaskCollator(
    mask_token_id=tokenizer.mask_token_id,
    pad_token_id=tokenizer.pad_token_id,
    num_target_spans=getattr(jepa_cfg, 'num_target_spans', 4),
    target_span_length=getattr(jepa_cfg, 'target_span_length', 8),
  )

  train_loader = valid_loader = None

  if not skip_train:
    train_loader = torch.utils.data.DataLoader(
    dataset_dict['train'],
    batch_size=config.loader.batch_size,
    num_workers=config.loader.num_workers,
    pin_memory=config.loader.pin_memory,
    shuffle=True,
    persistent_workers=config.loader.num_workers > 0,  # ← fix
    collate_fn=collator,
    )
    train_loader.tokenizer = tokenizer

  if not skip_valid:
    generator    = torch.Generator().manual_seed(valid_seed) if valid_seed else None
    valid_loader = torch.utils.data.DataLoader(
    dataset_dict['validation'],
    batch_size=config.loader.eval_batch_size,
    num_workers=config.loader.num_workers,
    pin_memory=config.loader.pin_memory,
    shuffle=valid_seed is not None,
    generator=generator,
    persistent_workers=config.loader.num_workers > 0,  # ← fix
    collate_fn=collator,
    )
    valid_loader.tokenizer = tokenizer

  return train_loader, valid_loader



In [11]:
# data collator

import typing

# ---------------------------------------------------------------------------
# JEPA mask collator
# ---------------------------------------------------------------------------

class JEPAMaskCollator:
  """Produces context / target view pairs for JEPA phase-1 training.

  Both views come from the same tokenized sequence:
    - target_input_ids:   clean, unmasked sequence  → target encoder (stop-grad)
    - context_input_ids:  target spans replaced with [MASK] → context encoder
    - target_mask:        bool (B, L), True at masked positions → predictor loss

  Span sampling: ``num_target_spans`` non-overlapping spans of
  ``target_span_length`` tokens, sampled uniformly from the maskable region
  (excluding BOS, EOS, and padding).
  """

  def __init__(
      self,
      mask_token_id: int,
      pad_token_id: int,
      num_target_spans: int = 4,
      target_span_length: int = 8,
  ):
    self.mask_token_id = mask_token_id
    self.pad_token_id = pad_token_id
    self.num_target_spans = num_target_spans
    self.target_span_length = target_span_length

  def __call__(
      self,
      batch: typing.List[typing.Dict[str, torch.Tensor]],
  ) -> typing.Dict[str, torch.Tensor]:
    input_ids      = torch.stack([b['input_ids']      for b in batch])
    attention_mask = torch.stack([b['attention_mask'] for b in batch])
    B, L = input_ids.shape

    target_input_ids  = input_ids.clone()
    context_input_ids = input_ids.clone()
    target_mask = torch.zeros(B, L, dtype=torch.bool)

    for i in range(B):
      valid_len = int(attention_mask[i].sum().item())
      # maskable region: skip BOS (idx 0) and EOS (last real token)
      spans = self._sample_spans(maskable_start=1, maskable_end=valid_len - 1)
      for s, e in spans:
        target_mask[i, s:e] = True
      context_input_ids[i, target_mask[i]] = self.mask_token_id

    return {
      'context_input_ids':      context_input_ids,   # (B, L)
      'context_attention_mask': attention_mask,       # (B, L)
      'target_input_ids':       target_input_ids,     # (B, L)
      'target_attention_mask':  attention_mask,       # (B, L)
      'target_mask':            target_mask,          # (B, L) bool
    }

  def _sample_spans(
      self,
      maskable_start: int,
      maskable_end: int,
  ) -> typing.List[typing.Tuple[int, int]]:
    region_len = maskable_end - maskable_start
    if region_len <= 0:
      return []

    span_len  = min(self.target_span_length, region_len)
    available = list(range(maskable_start, maskable_end - span_len + 1))

    spans: typing.List[typing.Tuple[int, int]] = []
    for _ in range(self.num_target_spans):
      if not available:
        break
      idx = torch.randint(len(available), (1,)).item()
      s, e = available[idx], available[idx] + span_len
      spans.append((s, e))
      available = [x for x in available if (x + span_len <= s) or (x >= e)]

    return spans

In [12]:
# ============================================================
# CELL 0 — Fix dependencies
# Run this cell, then RESTART THE RUNTIME, then run again.
# ============================================================

# The real root cause: `sentencepiece` is not installed.
# transformers needs it for bert/tokenization_utils_sentencepiece.
# Do NOT downgrade transformers — sentence-transformers needs >=4.41.
# Just install the missing package and let pip resolve cleanly.

!pip install -q sentencepiece
!pip install -q "transformers>=4.41,<4.46" --upgrade   # stays compatible with sentence-transformers

# After this finishes, go to Runtime → Restart runtime, then re-run all cells.


# ============================================================
# DATA PIPELINE SMOKE TEST
# ============================================================

import textwrap
import torch

# ── Minimal config stub ──────────────────────────────────────────────────────
class _C:
  def __init__(self, d):
    for k, v in d.items():
      setattr(self, k, _C(v) if isinstance(v, dict) else v)

config = _C({
  'data': {
    'tokenizer_name_or_path': 'bert-base-uncased',
    'cache_dir': '/content/notebooks_meta/v3/first_phase/.cache',
  },
  'model':  {'length': 128},
  'loader': {
    'batch_size': 8,
    'eval_batch_size': 8,
    'global_batch_size': 8,
    'eval_global_batch_size': 8,
    'num_workers': 0,
    'pin_memory': False,
  },
  'jepa': {
    'num_target_spans': 4,
    'target_span_length': 8,
  },
  'trainer': {
    'num_nodes': 1,
    'accumulate_grad_batches': 1,
  },
})

SEP  = '─' * 60
PASS = '✅'
FAIL = '❌'
SKIP = '⏭️ '

def section(title):
  print(f'\n{SEP}\n  {title}\n{SEP}')

errors = []

# Initialise shared variables — prevents NameError cascade if a stage fails
tokenizer    = None
dataset_dict = None
train_loader = None
valid_loader = None


# ────────────────────────────────────────────────────────────
# 1. ENVIRONMENT SANITY CHECK
# ────────────────────────────────────────────────────────────
section('0 · Environment versions')
try:
  import transformers, tokenizers, sentencepiece
  print(f'   torch          : {torch.__version__}')
  print(f'   transformers   : {transformers.__version__}')
  print(f'   tokenizers     : {tokenizers.__version__}')
  print(f'   sentencepiece  : {sentencepiece.__version__}')
  # Quick smoke-test that AutoTokenizer is importable
  from transformers import AutoTokenizer
  print(f'{PASS} All imports OK')
except Exception as e:
  errors.append(f'Environment: {e}')
  print(f'{FAIL} {e}')
  print('  → Run the pip install at the top of this cell, '
        'then Runtime → Restart runtime, then re-run.')
  import traceback; traceback.print_exc()


# ────────────────────────────────────────────────────────────
# 1. TOKENIZER
# ────────────────────────────────────────────────────────────
section('1 · Tokenizer')
try:
  tokenizer = get_tokenizer(config)
  print(f'{PASS} Loaded: {tokenizer.__class__.__name__}')
  print(f'   vocab_size  : {tokenizer.vocab_size}')
  print(f'   bos_token   : {tokenizer.bos_token!r}  (id {tokenizer.bos_token_id})')
  print(f'   eos_token   : {tokenizer.eos_token!r}  (id {tokenizer.eos_token_id})')
  print(f'   mask_token  : {tokenizer.mask_token!r} (id {tokenizer.mask_token_id})')
  print(f'   pad_token   : {tokenizer.pad_token!r}  (id {tokenizer.pad_token_id})')

  assert tokenizer.mask_token_id is not None, 'mask_token_id is None!'
  assert tokenizer.pad_token_id  is not None, 'pad_token_id is None!'
  print(f'{PASS} All required special tokens present')

  decoded = tokenizer.decode(
    tokenizer('Hello, how are you?', max_length=16,
              padding='max_length', truncation=True)['input_ids'],
    skip_special_tokens=False)
  print(f'   round-trip  : {decoded!r}')
except Exception as e:
  errors.append(f'Tokenizer: {e}')
  print(f'{FAIL} {e}')
  import traceback; traceback.print_exc()


# ────────────────────────────────────────────────────────────
# 2. DATASET
# ────────────────────────────────────────────────────────────
section('2 · Dataset — get_dailydialog_dataset')
if tokenizer is None:
  print(f'{SKIP} Skipped — tokenizer not available')
else:
  try:
    dataset_dict = get_dailydialog_dataset(
      cache_dir=config.data.cache_dir,
      tokenizer=tokenizer,
      block_size=config.model.length,
      num_proc=1,
    )
    for split, ds in dataset_dict.items():
      print(f'{PASS} split={split:12s}  rows={len(ds):>6,}  columns={ds.column_names}')

    ex    = dataset_dict['train'][0]
    ids   = ex['input_ids']
    amask = ex['attention_mask']
    print(f'\n   Sample (train[0]):')
    print(f'   input_ids shape      : {ids.shape}')
    print(f'   #real tokens         : {int(amask.sum())}')
    print(f'   decoded (truncated)  : '
          + textwrap.shorten(
              tokenizer.decode(ids, skip_special_tokens=False),
              width=80, placeholder=' ...'))

    assert ids.shape[0] == config.model.length
    num_pad  = int((ids == tokenizer.pad_token_id).sum())
    num_real = int(amask.sum())
    assert num_pad + num_real == config.model.length
    print(f'{PASS} Shape and padding checks passed')
  except Exception as e:
    errors.append(f'Dataset: {e}')
    print(f'{FAIL} {e}')
    import traceback; traceback.print_exc()


# ────────────────────────────────────────────────────────────
# 3. MASK COLLATOR (standalone)
# ────────────────────────────────────────────────────────────
section('3 · JEPAMaskCollator — standalone test')
if tokenizer is None or dataset_dict is None:
  print(f'{SKIP} Skipped — earlier stage failed')
else:
  try:
    collator  = JEPAMaskCollator(
      mask_token_id=tokenizer.mask_token_id,
      pad_token_id=tokenizer.pad_token_id,
      num_target_spans=config.jepa.num_target_spans,
      target_span_length=config.jepa.target_span_length,
    )
    raw_batch = [dataset_dict['train'][i] for i in range(4)]
    batch     = collator(raw_batch)

    expected_keys = {
      'context_input_ids', 'context_attention_mask',
      'target_input_ids',  'target_attention_mask',
      'target_mask',
    }
    assert set(batch.keys()) == expected_keys
    print(f'{PASS} Output keys correct')

    B, L = batch['context_input_ids'].shape
    print(f'   Batch shape : B={B}, L={L}')
    for name, t in batch.items():
      print(f'   {name:30s} {tuple(t.shape)}  {t.dtype}')

    ctx  = batch['context_input_ids']
    tgt  = batch['target_input_ids']
    mask = batch['target_mask']

    assert (ctx[mask]  == tokenizer.mask_token_id).all(), \
        'Context has non-[MASK] tokens at masked positions!'
    print(f'{PASS} Masked positions in context are all [MASK]')

    assert (tgt[mask]  != tokenizer.mask_token_id).all(), \
        'Target should be clean — no [MASK] tokens!'
    print(f'{PASS} Target is clean (no [MASK])')

    assert (ctx[~mask] == tgt[~mask]).all(), \
        'Context and target differ at unmasked positions!'
    print(f'{PASS} Context == target at unmasked positions')

    print()
    for i in range(B):
      valid  = int(batch['context_attention_mask'][i].sum())
      masked = int(mask[i].sum())
      print(f'   seq {i}: valid={valid:3d}  masked={masked:2d}  '
            f'ratio={masked/max(valid,1):.1%}')
  except Exception as e:
    errors.append(f'Collator: {e}')
    print(f'{FAIL} {e}')
    import traceback; traceback.print_exc()


# ────────────────────────────────────────────────────────────
# 4. FULL DATALOADER
# ────────────────────────────────────────────────────────────
section('4 · Full DataLoader — get_jepa_dataloaders')
if tokenizer is None or dataset_dict is None:
  print(f'{SKIP} Skipped — earlier stage failed')
else:
  try:
    train_loader, valid_loader = get_jepa_dataloaders(
      config, tokenizer, valid_seed=42)

    print(f'{PASS} train_loader : {len(train_loader):,} batches '
          f'(~{len(train_loader.dataset):,} samples)')
    print(f'{PASS} valid_loader : {len(valid_loader):,} batches '
          f'(~{len(valid_loader.dataset):,} samples)')

    for loader_name, loader in [('train', train_loader), ('valid', valid_loader)]:
      batch = next(iter(loader))
      print(f'\n   [{loader_name}] first batch:')
      for k, v in batch.items():
        print(f'     {k:30s} {tuple(v.shape)}  {v.dtype}')

      assert not torch.isnan(batch['context_input_ids'].float()).any()
      assert batch['target_mask'].any(), 'target_mask is all False!'
      ratio = batch['target_mask'].float().mean().item()
      assert 0.05 < ratio < 0.70, f'Suspicious mask ratio: {ratio:.1%}'
      print(f'   {PASS} {loader_name} — mask_ratio={ratio:.1%}  OK')
  except Exception as e:
    errors.append(f'DataLoader: {e}')
    print(f'{FAIL} {e}')
    import traceback; traceback.print_exc()


# ────────────────────────────────────────────────────────────
# 5. VISUAL SPOT-CHECK
# ────────────────────────────────────────────────────────────
section('5 · Spot-check — decoded context vs target')
if train_loader is None:
  print(f'{SKIP} Skipped — earlier stage failed')
else:
  try:
    batch    = next(iter(train_loader))
    idx      = 0
    real_len = int(batch['context_attention_mask'][idx].sum())

    tgt_decoded = tokenizer.decode(
      batch['target_input_ids'][idx, :real_len], skip_special_tokens=False)
    ctx_decoded = tokenizer.decode(
      batch['context_input_ids'][idx, :real_len], skip_special_tokens=False)

    print('  TARGET (clean):')
    print('  ' + textwrap.fill(tgt_decoded, width=70, subsequent_indent='  '))
    print('\n  CONTEXT (masked):')
    print('  ' + textwrap.fill(ctx_decoded, width=70, subsequent_indent='  '))

    positions = batch['target_mask'][idx, :real_len].nonzero(as_tuple=False).flatten().tolist()
    print(f'\n  Masked positions : {positions}')
    print(f'  ({len(positions)} / {real_len} tokens = '
          f'{len(positions)/real_len:.1%} masked)')
    print(f'{PASS} Spot-check complete')
  except Exception as e:
    errors.append(f'Spot-check: {e}')
    print(f'{FAIL} {e}')
    import traceback; traceback.print_exc()


# ────────────────────────────────────────────────────────────
# SUMMARY
# ────────────────────────────────────────────────────────────
section('SUMMARY')
if not errors:
  print(f'{PASS} All stages passed — safe to proceed to training.')
else:
  print(f'{FAIL} {len(errors)} stage(s) failed:')
  for e in errors:
    print(f'   • {e}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.3 MB/s eta 0:00:00

────────────────────────────────────────────────────────────
  0 · Environment versions
────────────────────────────────────────────────────────────
   torch          : 2.5.1+cu124
   transformers   : 4.45.2
   tokenizers     : 0.20.3
   sentencepiece  : 0.2.1
✅ All imports OK

────────────────────────────────────────────────────────────
  1 · Tokenizer
────────────────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Tokenizing test:   0%|          | 0/7740 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/87170 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8069 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7740 [00:00<?, ? examples/s]

Saved to cache: /content/notebooks_meta/v3/first_phase/.cache/dailydialog_jepa_bs128
✅ split=train         rows=87,170  columns=['input_ids', 'attention_mask']
✅ split=validation    rows= 8,069  columns=['input_ids', 'attention_mask']
✅ split=test          rows= 7,740  columns=['input_ids', 'attention_mask']

   Sample (train[0]):
   input_ids shape      : torch.Size([128])
   #real tokens         : 16
   decoded (truncated)  : [CLS] say, jim, how about going for a few beers after dinner? [SEP] [PAD] ...
✅ Shape and padding checks passed

────────────────────────────────────────────────────────────
  3 · JEPAMaskCollator — standalone test
────────────────────────────────────────────────────────────
✅ Output keys correct
   Batch shape : B=4, L=128
   context_input_ids              (4, 128)  torch.int64
   context_attention_mask         (4, 128)  torch.int64
   target_input_ids               (4, 128)  torch.int64
   target_attention_mask          (4, 128)  torch.int64
   target_mask    

In [13]:
"""
Stage 1 JEPA Training — DailyDialog
"""

import os
import sys
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm

sys.path.insert(0, '/content/notebooks_meta/v3/first_phase')

from cog_arch.encoder import Encoder
from cog_arch.dm import DM
from losses import SquareLossSeq, VCLoss

# ── Config ────────────────────────────────────────────────────────────────────


HIDDEN_SIZE = 256
NUM_HEADS   = 4       # must divide HIDDEN_SIZE: 256/4 = 64 per head
NUM_LAYERS  = 4
MAX_SEQ_LEN = 128
MLP_RATIO   = 4.0
VOCAB_SIZE  = tokenizer.vocab_size

LR           = 1e-4
WEIGHT_DECAY = 0.05
N_EPOCHS     = 20
EMA_DECAY    = 0.996

# VCReg coefficients (standard values from VICReg paper)
STD_COEFF = 25.0
COV_COEFF = 1.0

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_DIR = '/content/notebooks_meta/v3/first_phase/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)


# ── Dataset ───────────────────────────────────────────────────────────────────



# ── Data — reuse loaders already built above ──────────────────────────────────
# train_loader and valid_loader are already defined from the smoke test cell
# no need to rebuild them



# ── Models ────────────────────────────────────────────────────────────────────



context_encoder = Encoder(
    vocab_size=VOCAB_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    max_seq_len=MAX_SEQ_LEN,
).to(DEVICE)

target_encoder = copy.deepcopy(context_encoder).to(DEVICE)
for p in target_encoder.parameters():
    p.requires_grad = False

predictor = DM(
    hidden_size=HIDDEN_SIZE,
    depth=NUM_LAYERS,
    num_heads=NUM_HEADS,
    mlp_ratio=MLP_RATIO,
    max_seq_len=MAX_SEQ_LEN,
).to(DEVICE)


# ── Sanity check shapes before training ───────────────────────────────────────

with torch.no_grad():
    dummy_ctx_ids  = torch.randint(0, VOCAB_SIZE, (2, MAX_SEQ_LEN)).to(DEVICE)
    dummy_pos      = torch.zeros(2, 8, dtype=torch.long).to(DEVICE)

    ctx_out  = context_encoder(dummy_ctx_ids)
    pred_out = predictor(ctx_out, dummy_pos, condition=None)

    print(f"context_encoder output : {ctx_out.shape}")   # (2, 128, 256)
    print(f"predictor output       : {pred_out.shape}")  # (2, 8,  256)
    assert ctx_out.shape  == (2, MAX_SEQ_LEN, HIDDEN_SIZE), f"encoder shape wrong: {ctx_out.shape}"
    assert pred_out.shape == (2, 8, HIDDEN_SIZE),           f"predictor shape wrong: {pred_out.shape}"
    print("✅ Shape check passed")

# ── Loss ──────────────────────────────────────────────────────────────────────

pred_loss_fn = SquareLossSeq()
vc_loss_fn   = VCLoss(std_coeff=STD_COEFF, cov_coeff=COV_COEFF)

# ── Optimizer ─────────────────────────────────────────────────────────────────

optimizer = AdamW(
    list(context_encoder.parameters()) + list(predictor.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

# ── Helpers ───────────────────────────────────────────────────────────────────

def update_ema(context_enc, target_enc, decay=EMA_DECAY):
    with torch.no_grad():
        for p_c, p_t in zip(context_enc.parameters(), target_enc.parameters()):
            p_t.data.mul_(decay).add_(p_c.data, alpha=1.0 - decay)

def extract_target_positions(target_mask):
    """target_mask (B, L) bool → (B, T) int64 position indices"""
    B = target_mask.shape[0]
    T = int(target_mask.sum(dim=1).max().item())
    positions = torch.zeros(B, T, dtype=torch.long, device=target_mask.device)
    for i in range(B):
        pos = target_mask[i].nonzero(as_tuple=False).squeeze(1)
        positions[i, :len(pos)] = pos
    return positions

def unpack(batch):
    return (
        batch['context_input_ids'].to(DEVICE),
        batch['context_attention_mask'].to(DEVICE),
        batch['target_input_ids'].to(DEVICE),
        batch['target_attention_mask'].to(DEVICE),
        batch['target_mask'].to(DEVICE),
    )

# ── Train / eval steps ────────────────────────────────────────────────────────

def train_step(batch):
    ctx_ids, ctx_mask, tgt_ids, tgt_attn_mask, target_mask = unpack(batch)
    target_positions = extract_target_positions(target_mask)

    ctx_embeds = context_encoder(ctx_ids, attention_mask=ctx_mask)

    with torch.no_grad():
        tgt_embeds_full = target_encoder(tgt_ids, attention_mask=tgt_attn_mask)

    tgt_embeds = torch.stack([
        tgt_embeds_full[i, target_positions[i]]
        for i in range(ctx_ids.shape[0])
    ])

    pred_embeds = predictor(
        context_embeds=ctx_embeds,
        target_positions=target_positions,
        condition=None,
    )

    p_loss              = pred_loss_fn(pred_embeds, tgt_embeds)
    v_loss, _, vc_dict  = vc_loss_fn(pred_embeds)
    loss                = p_loss + v_loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(context_encoder.parameters()) + list(predictor.parameters()), 1.0
    )
    optimizer.step()
    update_ema(context_encoder, target_encoder)

    return loss.item(), p_loss.item(), v_loss.item(), vc_dict


@torch.no_grad()
def eval_step(batch):
    ctx_ids, ctx_mask, tgt_ids, tgt_attn_mask, target_mask = unpack(batch)
    target_positions = extract_target_positions(target_mask)

    ctx_embeds      = context_encoder(ctx_ids, attention_mask=ctx_mask)
    tgt_embeds_full = target_encoder(tgt_ids,  attention_mask=tgt_attn_mask)
    tgt_embeds      = torch.stack([
        tgt_embeds_full[i, target_positions[i]] for i in range(ctx_ids.shape[0])
    ])
    pred_embeds = predictor(ctx_embeds, target_positions, condition=None)

    p_loss         = pred_loss_fn(pred_embeds, tgt_embeds)
    v_loss, _, _   = vc_loss_fn(pred_embeds)
    return (p_loss + v_loss).item(), p_loss.item(), v_loss.item()




# ── Checkpointing ─────────────────────────────────────────────────────────────

def save_checkpoint(epoch, step):
    path = os.path.join(CKPT_DIR, f'epoch_{epoch}.pt')
    torch.save({
        'epoch':           epoch,
        'step':            step,
        'context_encoder': context_encoder.state_dict(),
        'target_encoder':  target_encoder.state_dict(),
        'predictor':       predictor.state_dict(),
        'optimizer':       optimizer.state_dict(),
    }, path)
    print(f"  saved → {path}")

# ── Training loop ─────────────────────────────────────────────────────────────


import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Accumulate metrics during training ───────────────────────────────────────
# Replace run() with this version that records everything

history = {
    'train_loss': [], 'train_pred': [], 'train_vc': [],
    'val_loss':   [], 'val_pred':   [], 'val_vc':   [],
    'std_loss':   [], 'cov_loss':   [],   # per-step VC components
    'lr':         [],                      # per-step learning rate
}

def run():
    global_step = 0
    for epoch in range(N_EPOCHS):
        context_encoder.train()
        predictor.train()

        train_loss = train_pred = train_vc = 0.0
        epoch_std = epoch_cov = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS}")
        for batch in pbar:
            loss, p, v, vc_dict = train_step(batch)
            train_loss += loss; train_pred += p; train_vc += v
            epoch_std  += vc_dict['std_loss']
            epoch_cov  += vc_dict['cov_loss']

            history['lr'].append(scheduler.get_last_lr()[0])
            global_step += 1

            pbar.set_postfix(
                loss=f"{loss:.4f}", pred=f"{p:.4f}",
                std=f"{vc_dict['std_loss']:.4f}",
                cov=f"{vc_dict['cov_loss']:.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}",
            )

        context_encoder.eval()
        predictor.eval()
        val_loss = val_pred = val_vc = 0.0
        for batch in valid_loader:
            l, p, v = eval_step(batch)
            val_loss += l; val_pred += p; val_vc += v

        nt, nv = len(train_loader), len(valid_loader)

        history['train_loss'].append(train_loss / nt)
        history['train_pred'].append(train_pred / nt)
        history['train_vc'].append(train_vc   / nt)
        history['val_loss'].append(val_loss   / nv)
        history['val_pred'].append(val_pred   / nv)
        history['val_vc'].append(val_vc     / nv)
        history['std_loss'].append(epoch_std  / nt)
        history['cov_loss'].append(epoch_cov  / nt)

        print(
            f"Epoch {epoch+1:02d} | "
            f"train {train_loss/nt:.4f} (pred {train_pred/nt:.4f} vc {train_vc/nt:.4f}) | "
            f"val   {val_loss/nv:.4f}  (pred {val_pred/nv:.4f} vc {val_vc/nv:.4f})"
        )
        save_checkpoint(epoch + 1, global_step)
        plot_training(history)   # live update after each epoch

run()

context_encoder output : torch.Size([2, 128, 256])
predictor output       : torch.Size([2, 8, 256])
✅ Shape check passed


Epoch 1/20: 100%|██████████| 10897/10897 [07:46<00:00, 23.35it/s, cov=0.1252, loss=1.1148, pred=0.8139, std=0.0070]


Epoch 01 | train 0.7471 (pred 0.4974 vc 0.2497) | val   0.8173  (pred 0.5982 vc 0.2190)
  saved → /content/notebooks_meta/v3/first_phase/checkpoints/epoch_1.pt


Epoch 2/20: 100%|██████████| 10897/10897 [07:35<00:00, 23.92it/s, cov=0.0820, loss=0.5575, pred=0.4755, std=0.0000]


Epoch 02 | train 0.8081 (pred 0.5705 vc 0.2376) | val   0.7584  (pred 0.5368 vc 0.2216)
  saved → /content/notebooks_meta/v3/first_phase/checkpoints/epoch_2.pt


...

Epoch 18 | train 0.8476 (pred 0.6342 vc 0.2134) | val   0.8404  (pred 0.6388 vc 0.2016)
  saved → /content/notebooks_meta/v3/first_phase/checkpoints/epoch_18.pt


Epoch 19/20: 100%|██████████| 10897/10897 [07:47<00:00, 23.33it/s, cov=0.0204, loss=0.6717, pred=0.6221, std=0.0012]


Epoch 19 | train 0.8595 (pred 0.6451 vc 0.2145) | val   0.8629  (pred 0.6467 vc 0.2161)
  saved → /content/notebooks_meta/v3/first_phase/checkpoints/epoch_19.pt


Epoch 20/20: 100%|██████████| 10897/10897 [07:50<00:00, 23.17it/s, cov=0.0925, loss=0.8330, pred=0.7405, std=0.0000]


Epoch 20 | train 0.8682 (pred 0.6563 vc 0.2118) | val   0.8695  (pred 0.6586 vc 0.2109)
  saved → /content/notebooks_meta/v3/first_phase/checkpoints/epoch_20.pt


In [ ]:
# plotting

def plot_training(history, save_path=None):
    epochs = range(1, len(history['train_loss']) + 1)
    steps  = range(len(history['lr']))

    fig = plt.figure(figsize=(16, 12))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3)

    # ── 1. Total loss: train vs val ───────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(epochs, history['train_loss'], 'b-o', label='train', markersize=4)
    ax1.plot(epochs, history['val_loss'],   'r-o', label='val',   markersize=4)
    ax1.set_title('Total Loss')
    ax1.set_xlabel('Epoch')
    ax1.legend()
    ax1.grid(alpha=0.3)

    # ── 2. Prediction loss: train vs val ─────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(epochs, history['train_pred'], 'b-o', label='train', markersize=4)
    ax2.plot(epochs, history['val_pred'],   'r-o', label='val',   markersize=4)
    ax2.set_title('Prediction Loss  (lower = better JEPA predictions)')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    ax2.grid(alpha=0.3)

    # ── 3. VC regularization components ──────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.plot(epochs, history['std_loss'], 'g-o', label='std loss', markersize=4)
    ax3.plot(epochs, history['cov_loss'], 'm-o', label='cov loss', markersize=4)
    ax3.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    ax3.set_title('VCReg Components  (std→1, cov→0 = healthy)')
    ax3.set_xlabel('Epoch')
    ax3.legend()
    ax3.grid(alpha=0.3)

    # ── 4. Train vs val VC loss ───────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.plot(epochs, history['train_vc'], 'b-o', label='train vc', markersize=4)
    ax4.plot(epochs, history['val_vc'],   'r-o', label='val vc',   markersize=4)
    ax4.set_title('VC Regularization Loss')
    ax4.set_xlabel('Epoch')
    ax4.legend()
    ax4.grid(alpha=0.3)

    # ── 5. Train/val gap (overfitting detector) ───────────────────────────────
    ax5 = fig.add_subplot(gs[2, 0])
    gap = [v - t for t, v in zip(history['train_loss'], history['val_loss'])]
    colors = ['red' if g > 0 else 'green' for g in gap]
    ax5.bar(epochs, gap, color=colors, alpha=0.7)
    ax5.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    ax5.set_title('Val − Train Gap  (red = overfitting, green = underfitting)')
    ax5.set_xlabel('Epoch')
    ax5.grid(alpha=0.3)

    # ── 6. Learning rate schedule ─────────────────────────────────────────────
    ax6 = fig.add_subplot(gs[2, 1])
    ax6.plot(steps, history['lr'], 'orange', linewidth=1, alpha=0.8)
    ax6.set_title('Learning Rate Schedule')
    ax6.set_xlabel('Step')
    ax6.set_yscale('log')
    ax6.grid(alpha=0.3)

    plt.suptitle(
        f'JEPA Stage 1 Training  —  epoch {len(epochs)}/{N_EPOCHS}',
        fontsize=13, fontweight='bold'
    )

    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')

    plt.show()
    plt.close()

In [14]:
!ls /content/notebooks_meta/v3/first_phase

checkpoints  data    first_phase_01.ipynb  losses.py	train_utils.py
cog_arch     ema.py  __init__.py	   __pycache__
